# Lesson 02 - Explorarea Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** este un cadru unificat pentru construirea agenților AI. Oferă o arhitectură clară, componibilă, cu patru blocuri de bază:

- **Client** – se conectează la un endpoint de model AI și gestionează comunicarea
- **Agent** – înfășoară un client cu instrucțiuni și definiții de unelte
- **Unelte** – extind capacitățile agentului cu funcții personalizate pe care modelul le poate apela
- **Sesiune** – păstrează istoricul conversației pentru interacțiuni multi-turn

În această lecție, vom construi un **agent de rezervare a călătoriilor** care verifică disponibilitatea destinației folosind aceste concepte.


## Configurare


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Înțelegerea arhitecturii Agent Framework

Microsoft Agent Framework urmează o arhitectură stratificată:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Un `AzureAIProjectAgentProvider` se conectează la o implementare Azure OpenAI. Acesta se ocupă de autentificare, formatarea cererilor și analiza răspunsurilor.
2. **Agent** – Creat din client prin `provider.create_agent()`, agentul combină accesul la model cu instrucțiuni (prompt de sistem) și unelte.
3. **Unelte** – Funcții Python decorate cu `@tool` pe care agentul le poate invoca pentru a realiza acțiuni sau a obține date.
4. **Sesiune** – Un obiect `AgentSession` (creat prin `agent.create_session()`) care stochează istoricul conversației, permițând dialoguri multi-turn unde agentul își amintește contextul anterior.

Să construim fiecare strat pas cu pas.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adăugarea uneltelor cu decoratorul @tool

Uneltele permit agenților să întreprindă acțiuni dincolo de generarea de text. Decoratorul `@tool` transformă o funcție Python obișnuită într-un ceva ce agentul poate apela.

Puncte cheie:
- Folosiți `Annotated[type, "description"]` pentru ca modelul să înțeleagă fiecare parametru.
- Docstring-ul devine descrierea uneltei pe care o vede modelul.
- `approval_mode="never_require"` înseamnă că unealta rulează automat fără confirmarea utilizatorului.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Crearea unui Agent cu Instrumente

Acum combinăm clientul, instrucțiunile și instrumentele într-un agent. `instructions` acționează ca promptul sistemului — ele definesc persoana și comportamentul agentului.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Convorbiri cu mai multe schimburi folosind Sesiuni

Un `AgentSession` (creat prin `agent.create_session()`) urmărește toate mesajele dintr-o conversație. Prin transmiterea aceleiași sesiuni fiecărei apelări `agent.run()`, agentul are acces la întregul istoric al conversației și se poate referi la mesajele anterioare.

Transmitem `tools=[check_destination_availability]` astfel încât agentul să poată apela verificatorul nostru de disponibilitate în fiecare schimb.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Rezumat

În această lecție ai explorat cele patru piloni ai Microsoft Agent Framework:

| Concept | Ce ai învățat |
|---------|---------------|
| **Client** | `AzureAIProjectAgentProvider` se conectează la Azure OpenAI cu autentificare bazată pe acreditări |
| **Agent** | `provider.create_agent()` combină o conexiune la model cu instrucțiuni și un nume |
| **Unelte** | Decoratorul `@tool` expune funcții Python pentru ca agentul să le poată apela |
| **Sesiune** | `agent.create_session()` menține istoricul conversației pe mai multe runde |

Aceste componente construiesc agenți care pot susține conversații naturale, pot apela funcții externe și pot menține contextul — fundamentul pentru modele agentice mai avansate în lecțiile următoare.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare de responsabilitate**:  
Acest document a fost tradus folosind serviciul de traducere automată AI [Co-op Translator](https://github.com/Azure/co-op-translator). Deși ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original, în limba sa nativă, trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un specialist uman. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite rezultate din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
